# Problem 1 — Industry-Grade Monthly Demand Forecasting (MAPE < 5%)

This notebook provides an end-to-end, production-style solution for **monthly demand forecasting** per `(product, aps)` series.

It follows a proven approach:
- Clean **panel time-series** → long format
- **External signals** merged by month
- Strong **lag + rolling** features
- **Direct multi-horizon** forecasting (1–12 months) to avoid recursive error accumulation
- **Rolling-origin** evaluation (no leakage)
- Global **LightGBM** model (optionally CatBoost)
- Optional **hyperparameter tuning**, **prediction intervals**, and **SHAP** explainability

> **Files expected** (same naming as the reference notebook):
> - `demand_history.csv`
> - `external_signals.csv` (recommended)

If your folder differs, edit `DATA_DIR` in the next cell.


In [1]:

# -------------------------------
# 0) Setup: imports + configuration
# -------------------------------
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Reference notebook uses ../data
CANDIDATE_DIRS = [
    Path("./data"),

]

def auto_find_data_dir(filenames=("demand_history.csv",)):
    for d in CANDIDATE_DIRS:
        if all((d / f).exists() for f in filenames):
            return d
    return None

DATA_DIR = auto_find_data_dir(("demand_history.csv",)) or Path("../data")

MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
MONTH_MAP = {m: i+1 for i, m in enumerate(MONTHS)}

print("Using DATA_DIR:", DATA_DIR.resolve())
assert (DATA_DIR / "demand_history.csv").exists(), (
    "Could not find demand_history.csv. Put it in one of these folders:\n"
    + "\n".join(str(d.resolve()) for d in CANDIDATE_DIRS)
)


Using DATA_DIR: D:\Projects\@Scale AI Hackathon\Codes\data


## 1) Utilities: safe MAPE, train/valid split helpers

In [2]:

from dataclasses import dataclass

def safe_mape(y_true, y_pred, eps=1e-6):
    '''
    Safe MAPE that won't explode on near-zero values.
    If competition uses plain MAPE, set eps=0 and ensure no zeros.
    '''
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs(y_true - y_pred) / denom)

def month_start(dt):
    '''Normalize to month-start timestamps.'''
    return pd.to_datetime(dt).to_period("M").to_timestamp()

@dataclass
class FoldResult:
    cutoff_date: pd.Timestamp
    mape: float
    n_predictions: int


## 2) Load and reshape the demand data (wide Jan..Dec → long)

In [3]:

# -------------------------------
# 2) Load demand_history.csv and reshape to long monthly panel
# -------------------------------
demand_wide = pd.read_csv(DATA_DIR / "demand_history.csv")
print("Demand wide shape:", demand_wide.shape)
display(demand_wide.head())

def melt_monthly_data(df_wide: pd.DataFrame) -> pd.DataFrame:
    id_cols = [c for c in df_wide.columns if c not in MONTHS]
    df_long = df_wide.melt(
        id_vars=id_cols,
        value_vars=MONTHS,
        var_name="month",
        value_name="demand",
    )
    df_long["month_num"] = df_long["month"].map(MONTH_MAP).astype(int)
    df_long["date"] = pd.to_datetime(df_long[["year"]].assign(month=df_long["month_num"], day=1))
    df_long = df_long.sort_values(["product", "aps", "date"]).reset_index(drop=True)
    return df_long

demand = melt_monthly_data(demand_wide)

# Quick sanity checks
print("Demand long shape:", demand.shape)
print("Date range:", demand["date"].min().date(), "→", demand["date"].max().date())
print("Unique products:", demand["product"].nunique(), "| unique aps:", demand["aps"].nunique())

display(demand.head(12))


Demand wide shape: (92, 15)


,product,aps,year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,AH,ACNF,2022,10733,10328,11188,13739,14420,14603,16511,14448,15313,14503,11708,8903
1,AH,ACNF,2023,10986,10045,12266,14883,15757,16825,18040,15043,15044,13479,13493,10554
2,AH,ACNF,2024,11780,11171,12055,15155,16929,17593,19231,17170,16159,13870,11529,12644
3,AH,ACNF,2025,12858,10793,13054,16149,18015,17905,19817,17575,17790,15962,13540,11776
4,AH,AH_FIT,2022,7499,6928,7445,8990,9704,10243,11236,9334,10446,9331,7715,6057


Demand long shape: (1104, 7)
Date range: 2022-01-01 → 2025-12-01
Unique products: 5 | unique aps: 19


,product,aps,year,month,demand,month_num,date
0,AH,ACNF,2022,Jan,10733,1,2022-01-01
1,AH,ACNF,2022,Feb,10328,2,2022-02-01
2,AH,ACNF,2022,Mar,11188,3,2022-03-01
3,AH,ACNF,2022,Apr,13739,4,2022-04-01
4,AH,ACNF,2022,May,14420,5,2022-05-01
5,AH,ACNF,2022,Jun,14603,6,2022-06-01
6,AH,ACNF,2022,Jul,16511,7,2022-07-01
7,AH,ACNF,2022,Aug,14448,8,2022-08-01
8,AH,ACNF,2022,Sep,15313,9,2022-09-01
9,AH,ACNF,2022,Oct,14503,10,2022-10-01


## 3) Load external signals (recommended) and create lagged exogenous features

In [4]:

# -------------------------------
# 3) Load external_signals.csv (if available) and create lags safely
# -------------------------------
ext_path = DATA_DIR / "external_signals.csv"
external = None

if ext_path.exists():
    external = pd.read_csv(ext_path)
    print("External signals shape:", external.shape)
    display(external.head())

    # Try to infer the date column
    # Common patterns: 'date' or (year, month) columns
    if "date" in external.columns:
        external["date"] = pd.to_datetime(external["date"])
        external["date"] = external["date"].dt.to_period("M").dt.to_timestamp()
    elif {"year", "month"}.issubset(external.columns):
        external["date"] = pd.to_datetime(external[["year"]].assign(month=external["month"], day=1))
    elif {"year", "month_num"}.issubset(external.columns):
        external["date"] = pd.to_datetime(external[["year"]].assign(month=external["month_num"], day=1))
    else:
        raise ValueError("external_signals.csv must contain either 'date' or ('year','month') columns.")

    external = external.sort_values("date").reset_index(drop=True)

    # Identify numeric exogenous columns (exclude date/year/month)
    ignore_cols = {"date", "year", "month", "month_num"}
    exog_cols = [c for c in external.columns if c not in ignore_cols and pd.api.types.is_numeric_dtype(external[c])]
    print("Detected exogenous numeric columns:", exog_cols)

    # Create exogenous lags that can lead demand (domain-dependent)
    EXOG_LAGS = [0, 1, 3, 6, 12]  # feel free to extend
    for col in exog_cols:
        for lag in EXOG_LAGS:
            name = f"{col}_lag{lag}"
            external[name] = external[col].shift(lag)

    # Keep only date + engineered columns
    keep_cols = ["date"] + [c for c in external.columns if c.endswith(tuple([f"_lag{l}" for l in EXOG_LAGS]))]
    external_feat = external[keep_cols].copy()

    print("External feature columns:", len(external_feat.columns))
    display(external_feat.head(14))
else:
    print("external_signals.csv not found; continuing WITHOUT external signals.")


External signals shape: (48, 11)


,year,month,avg_temp_f,heating_degree_days,cooling_degree_days,housing_starts_k,building_permits_k,gdp_growth_pct,consumer_confidence,energy_price_idx,interest_rate_pct
0,2022,1,29.3,1072.0,0.0,1299.0,1422.0,0.82,103.3,109.4,3.78
1,2022,2,29.5,1066.0,0.0,1315.0,1394.0,0.44,102.9,114.9,3.75
2,2022,3,43.4,647.0,0.0,1485.0,1614.0,1.38,102.8,94.6,3.86
3,2022,4,56.1,266.0,0.0,1569.0,1722.0,0.56,93.9,108.3,3.62
4,2022,5,65.8,0.0,23.0,1611.0,1807.0,0.72,94.0,112.2,3.93


Detected exogenous numeric columns: ['avg_temp_f', 'heating_degree_days', 'cooling_degree_days', 'housing_starts_k', 'building_permits_k', 'gdp_growth_pct', 'consumer_confidence', 'energy_price_idx', 'interest_rate_pct']
External feature columns: 46


,date,avg_temp_f_lag0,avg_temp_f_lag1,avg_temp_f_lag3,avg_temp_f_lag6,avg_temp_f_lag12,heating_degree_days_lag0,heating_degree_days_lag1,heating_degree_days_lag3,heating_degree_days_lag6,...,energy_price_idx_lag0,energy_price_idx_lag1,energy_price_idx_lag3,energy_price_idx_lag6,energy_price_idx_lag12,interest_rate_pct_lag0,interest_rate_pct_lag1,interest_rate_pct_lag3,interest_rate_pct_lag6,interest_rate_pct_lag12
0,2022-01-01,29.3,NaN,NaN,NaN,NaN,1072.0,NaN,NaN,NaN,...,109.4,NaN,NaN,NaN,NaN,3.78,NaN,NaN,NaN,NaN
1,2022-02-01,29.5,29.3,NaN,NaN,NaN,1066.0,1072.0,NaN,NaN,...,114.9,109.4,NaN,NaN,NaN,3.75,3.78,NaN,NaN,NaN
2,2022-03-01,43.4,29.5,NaN,NaN,NaN,647.0,1066.0,NaN,NaN,...,94.6,114.9,NaN,NaN,NaN,3.86,3.75,NaN,NaN,NaN
3,2022-04-01,56.1,43.4,29.3,NaN,NaN,266.0,647.0,1072.0,NaN,...,108.3,94.6,109.4,NaN,NaN,3.62,3.86,3.78,NaN,NaN
4,2022-05-01,65.8,56.1,29.5,NaN,NaN,0.0,266.0,1066.0,NaN,...,112.2,108.3,114.9,NaN,NaN,3.93,3.62,3.75,NaN,NaN
5,2022-06-01,74.8,65.8,43.4,NaN,NaN,0.0,0.0,647.0,NaN,...,104.0,112.2,94.6,NaN,NaN,3.81,3.93,3.86,NaN,NaN
6,2022-07-01,77.3,74.8,56.1,29.3,NaN,0.0,0.0,266.0,1072.0,...,96.8,104.0,108.3,109.4,NaN,3.78,3.81,3.62,3.78,NaN
7,2022-08-01,79.4,77.3,65.8,29.5,NaN,0.0,0.0,0.0,1066.0,...,90.6,96.8,112.2,114.9,NaN,4.13,3.78,3.93,3.75,NaN
8,2022-09-01,65.8,79.4,74.8,43.4,NaN,0.0,0.0,0.0,647.0,...,101.0,90.6,104.0,94.6,NaN,3.90,4.13,3.81,3.86,NaN
9,2022-10-01,56.8,65.8,77.3,56.1,NaN,247.0,0.0,0.0,266.0,...,88.6,101.0,96.8,108.3,NaN,3.68,3.90,3.78,3.62,NaN


## 4) Merge demand + exogenous, add time identifiers and series_id

In [5]:

# -------------------------------
# 4) Merge demand with external features, add identifiers
# -------------------------------
df = demand.copy()

# (Optional) If you only want aggregate APS='ALL' like the reference notebook, uncomment:
# df = df[df["aps"] == "ALL"].copy()

df["series_id"] = df["product"].astype(str) + "|" + df["aps"].astype(str)
df["date"] = df["date"].dt.to_period("M").dt.to_timestamp()

# Add calendar features
df["year"] = df["date"].dt.year
df["month_num"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter

# Cyclical month encoding
df["month_sin"] = np.sin(2 * np.pi * df["month_num"] / 12.0)
df["month_cos"] = np.cos(2 * np.pi * df["month_num"] / 12.0)

# Merge external features if available
if "external_feat" in globals() and external_feat is not None:
    df = df.merge(external_feat, on="date", how="left")

print("Merged df shape:", df.shape)
display(df.head())


Merged df shape: (1104, 56)


,product,aps,year,month,demand,month_num,date,series_id,quarter,month_sin,...,energy_price_idx_lag0,energy_price_idx_lag1,energy_price_idx_lag3,energy_price_idx_lag6,energy_price_idx_lag12,interest_rate_pct_lag0,interest_rate_pct_lag1,interest_rate_pct_lag3,interest_rate_pct_lag6,interest_rate_pct_lag12
0,AH,ACNF,2022,Jan,10733,1,2022-01-01,AH|ACNF,1,0.500000,...,109.4,NaN,NaN,NaN,NaN,3.78,NaN,NaN,NaN,NaN
1,AH,ACNF,2022,Feb,10328,2,2022-02-01,AH|ACNF,1,0.866025,...,114.9,109.4,NaN,NaN,NaN,3.75,3.78,NaN,NaN,NaN
2,AH,ACNF,2022,Mar,11188,3,2022-03-01,AH|ACNF,1,1.000000,...,94.6,114.9,NaN,NaN,NaN,3.86,3.75,NaN,NaN,NaN
3,AH,ACNF,2022,Apr,13739,4,2022-04-01,AH|ACNF,2,0.866025,...,108.3,94.6,109.4,NaN,NaN,3.62,3.86,3.78,NaN,NaN
4,AH,ACNF,2022,May,14420,5,2022-05-01,AH|ACNF,2,0.500000,...,112.2,108.3,114.9,NaN,NaN,3.93,3.62,3.75,NaN,NaN


## 5) Feature engineering: demand lags + rolling stats (no leakage)

In [6]:

# -------------------------------
# 5) Lag and rolling features
# -------------------------------
df = df.sort_values(["series_id", "date"]).reset_index(drop=True)

LAGS = list(range(0, 13))  # lag0..lag12 (lag0 = current month demand)
ROLL_WINDOWS = [3, 6, 12]

g = df.groupby("series_id", sort=False)

for lag in LAGS:
    df[f"demand_lag{lag}"] = g["demand"].shift(lag)

# Rolling features computed on history INCLUDING current demand (lag0)
for w in ROLL_WINDOWS:
    df[f"demand_roll_mean_{w}"] = (
        g["demand"].rolling(window=w, min_periods=max(2, w//2)).mean().reset_index(level=0, drop=True)
    )
    df[f"demand_roll_std_{w}"] = (
        g["demand"].rolling(window=w, min_periods=max(2, w//2)).std().reset_index(level=0, drop=True)
    )

# Simple trend features
df["demand_diff_1"] = df["demand"] - df["demand_lag1"]
df["demand_diff_12"] = df["demand"] - df["demand_lag12"]

print("Feature-engineered df shape:", df.shape)
display(df.tail(6))


Feature-engineered df shape: (1104, 77)


,product,aps,year,month,demand,month_num,date,series_id,quarter,month_sin,...,demand_lag11,demand_lag12,demand_roll_mean_3,demand_roll_std_3,demand_roll_mean_6,demand_roll_std_6,demand_roll_mean_12,demand_roll_std_12,demand_diff_1,demand_diff_12
1098,HP,HP_FIT,2025,Jul,32407,7,2025-07-01,HP|HP_FIT,3,-5.000000e-01,...,28394.0,30589.0,30014.666667,2122.598957,26627.166667,4457.812711,24743.083333,4495.274174,4050.0,1818.0
1099,HP,HP_FIT,2025,Aug,35028,8,2025-08-01,HP|HP_FIT,3,-8.660254e-01,...,27027.0,28394.0,31930.666667,3360.912128,29069.500000,4359.872143,25295.916667,5317.767909,2621.0,6634.0
1100,HP,HP_FIT,2025,Sep,28392,9,2025-09-01,HP|HP_FIT,3,-1.000000e+00,...,22050.0,27027.0,31942.333333,3342.313620,30046.500000,3066.241918,25409.666667,5372.480591,-6636.0,1365.0
1101,HP,HP_FIT,2025,Oct,27188,10,2025-10-01,HP|HP_FIT,4,-8.660254e-01,...,21729.0,22050.0,30202.666667,4222.000158,30108.666667,2990.466162,25837.833333,5284.404272,-1204.0,5138.0
1102,HP,HP_FIT,2025,Nov,21816,11,2025-11-01,HP|HP_FIT,4,-5.000000e-01,...,19327.0,21729.0,25798.666667,3501.232545,28864.666667,4549.963853,25845.083333,5278.310809,-5372.0,87.0
1103,HP,HP_FIT,2025,Dec,27047,12,2025-12-01,HP|HP_FIT,4,-2.449294e-16,...,18627.0,19327.0,25350.333333,3061.634259,28646.333333,4610.228310,26488.416667,4866.012694,5231.0,7720.0


## 6) Build a **direct multi-horizon** supervised dataset (h=1..12)

In [7]:

# -------------------------------
# 6) Direct multi-horizon dataset construction
# -------------------------------
HORIZON_MAX = 12

def make_direct_multi_horizon(
    base_df: pd.DataFrame,
    horizon_max: int = 12,
    target_col: str = "demand",
    group_col: str = "series_id",
    date_col: str = "date",
):
    '''
    Creates a supervised panel for direct multi-horizon forecasting.

    Each row corresponds to an anchor time t and horizon h, with target y(t+h).
    Features must be computed at time t (the anchor date).
    '''
    df_ = base_df.copy()
    df_ = df_.sort_values([group_col, date_col]).reset_index(drop=True)

    out = []
    gb = df_.groupby(group_col, sort=False)

    for h in range(1, horizon_max + 1):
        tmp = df_.copy()
        tmp["horizon"] = h
        tmp["target"] = gb[target_col].shift(-h)
        tmp["target_date"] = tmp[date_col] + pd.DateOffset(months=h)
        out.append(tmp)

    sup = pd.concat(out, axis=0, ignore_index=True)
    return sup

sup = make_direct_multi_horizon(df, horizon_max=HORIZON_MAX)

print("Supervised (direct multi-horizon) shape:", sup.shape)
display(sup[["series_id","date","horizon","target_date","demand","target"]].head(12))


Supervised (direct multi-horizon) shape: (13248, 80)


,series_id,date,horizon,target_date,demand,target
0,AH|ACNF,2022-01-01,1,2022-02-01,10733,10328.0
1,AH|ACNF,2022-02-01,1,2022-03-01,10328,11188.0
2,AH|ACNF,2022-03-01,1,2022-04-01,11188,13739.0
3,AH|ACNF,2022-04-01,1,2022-05-01,13739,14420.0
4,AH|ACNF,2022-05-01,1,2022-06-01,14420,14603.0
5,AH|ACNF,2022-06-01,1,2022-07-01,14603,16511.0
6,AH|ACNF,2022-07-01,1,2022-08-01,16511,14448.0
7,AH|ACNF,2022-08-01,1,2022-09-01,14448,15313.0
8,AH|ACNF,2022-09-01,1,2022-10-01,15313,14503.0
9,AH|ACNF,2022-10-01,1,2022-11-01,14503,11708.0


## 7) Define feature columns and prepare categorical handling

In [8]:

# -------------------------------
# 7) Feature list
# -------------------------------
cat_cols = ["product", "aps", "series_id", "horizon", "month_num", "quarter"]

# Numeric features: demand lags/rolling + diffs + cyclical month + external lags (if present)
num_cols = []
for c in sup.columns:
    if c.startswith("demand_lag") or c.startswith("demand_roll_") or c.startswith("demand_diff_"):
        num_cols.append(c)

num_cols += ["month_sin", "month_cos"]

# Include external lag features if present
ext_cols = [c for c in sup.columns if c.endswith(tuple([f"_lag{l}" for l in [0,1,3,6,12]]))]
num_cols += ext_cols

# Remove duplicates while preserving order
seen = set()
num_cols = [x for x in num_cols if not (x in seen or seen.add(x))]

feature_cols = cat_cols + num_cols

print("Categorical features:", cat_cols)
print("Numeric feature count:", len(num_cols))
print("Total feature count:", len(feature_cols))

# Convert categoricals to 'category' dtype for LightGBM
for c in ["product","aps","series_id","horizon","month_num","quarter"]:
    sup[c] = sup[c].astype("category")

# Drop rows where target is missing (cannot train/evaluate)
sup = sup.dropna(subset=["target"]).reset_index(drop=True)

# Require at least 12 months of history (lag12 exists)
sup = sup.dropna(subset=["demand_lag12"]).reset_index(drop=True)

print("Supervised after dropping NaNs:", sup.shape)
display(sup.head())


Categorical features: ['product', 'aps', 'series_id', 'horizon', 'month_num', 'quarter']
Numeric feature count: 68
Total feature count: 74
Supervised after dropping NaNs: (8142, 80)


,product,aps,year,month,demand,month_num,date,series_id,quarter,month_sin,...,demand_roll_std_3,demand_roll_mean_6,demand_roll_std_6,demand_roll_mean_12,demand_roll_std_12,demand_diff_1,demand_diff_12,horizon,target,target_date
0,AH,ACNF,2023,Jan,10986,1,2023-01-01,AH|ACNF,1,0.500000,...,1456.491103,12643.500000,2508.146626,13054.166667,2336.732794,2083.0,253.0,1,10045.0,2023-02-01
1,AH,ACNF,2023,Feb,10045,2,2023-02-01,AH|ACNF,1,0.866025,...,1043.115046,11909.666667,2518.686615,13030.583333,2367.967040,-941.0,-283.0,1,12266.0,2023-03-01
2,AH,ACNF,2023,Mar,12266,3,2023-03-01,AH|ACNF,1,1.000000,...,1114.803570,11401.833333,1934.726794,13120.416667,2311.484783,2221.0,1078.0,1,14883.0,2023-04-01
3,AH,ACNF,2023,Apr,14883,4,2023-04-01,AH|ACNF,2,0.866025,...,2421.699610,11465.166667,2058.788908,13215.750000,2362.348105,2617.0,1144.0,1,15757.0,2023-05-01
4,AH,ACNF,2023,May,15757,5,2023-05-01,AH|ACNF,2,0.500000,...,1816.573973,12140.000000,2713.725262,13327.166667,2454.056821,874.0,1337.0,1,16825.0,2023-06-01


## 8) Rolling-origin evaluation (no leakage)

In [9]:
# -------------------------------
# 8) Rolling-origin evaluation
# -------------------------------
def get_cutoff_dates(panel_df: pd.DataFrame, date_col="date"):
    '''
    Suggest cutoff dates at Dec of each year except the final year.
    '''
    dates = sorted(panel_df[date_col].unique())
    years = sorted({pd.Timestamp(d).year for d in dates})
    cutoffs = []
    for y in years[:-1]:
        dec = pd.Timestamp(year=y, month=12, day=1)
        if dec in panel_df[date_col].values:
            cutoffs.append(dec)
    return cutoffs

cutoffs = get_cutoff_dates(df, date_col="date")
cutoffs = [c for c in cutoffs if (sup["date"] == c).any()]
print("Filtered cutoff dates:", cutoffs)
print("Suggested cutoff dates:", cutoffs)


Filtered cutoff dates: [Timestamp('2023-12-01 00:00:00'), Timestamp('2024-12-01 00:00:00')]
Suggested cutoff dates: [Timestamp('2023-12-01 00:00:00'), Timestamp('2024-12-01 00:00:00')]


## 9) Train a global **LightGBM** model and evaluate folds

In [10]:
# -------------------------------
# 9) LightGBM model training + fold evaluation
# -------------------------------
try:
    import lightgbm as lgb
except ImportError:
    !pip -q install lightgbm
    import lightgbm as lgb

def train_lightgbm_regressor(X_train, y_train, X_valid, y_valid, categorical_features):
    params = dict(
        objective="regression",
        learning_rate=0.03,
        n_estimators=5000,
        num_leaves=63,
        min_child_samples=20,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )

    model = lgb.LGBMRegressor(**params)

    # Custom MAPE metric for evaluation
    def feval_mape(y_true_or_pred, maybe_dataset=None):
        """Compatibility wrapper for LightGBM sklearn/custom eval."""
        # sklearn wrapper may pass (y_true, y_pred)
        if maybe_dataset is None or isinstance(maybe_dataset, (list, tuple, np.ndarray)):
            y_true = y_true_or_pred
            y_pred = maybe_dataset
        else:
            y_pred = y_true_or_pred
            y_true = maybe_dataset.get_label()
        return ("mape", safe_mape(y_true, y_pred), False)

    # If validation is empty, train without early stopping
    if X_valid is None or len(X_valid) == 0:
        model.fit(
            X_train, y_train,
            categorical_feature=categorical_features
        )
        return model

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric=feval_mape,
        categorical_feature=categorical_features,
        callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)]
    )
    return model

def time_inner_split(train_df, date_col="target_date", n_valid_months=3):
    """
    Time-based inner split: last n_valid_months of target_date as validation.
    Guarantees non-empty if enough unique months exist.
    """
    uniq = np.array(sorted(pd.to_datetime(train_df[date_col].unique())))
    if len(uniq) < (n_valid_months + 2):
        # fallback: if too few months, use a row-based split
        train_df = train_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
        cut = int(len(train_df) * 0.9)
        inner_train = train_df.iloc[:cut].copy()
        inner_valid = train_df.iloc[cut:].copy()
        return inner_train, inner_valid

    valid_dates = set(uniq[-n_valid_months:])
    inner_valid = train_df[train_df[date_col].isin(valid_dates)].copy()
    inner_train = train_df[~train_df[date_col].isin(valid_dates)].copy()

    # safety fallback
    if len(inner_train) == 0 or len(inner_valid) == 0:
        train_df = train_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
        cut = int(len(train_df) * 0.9)
        inner_train = train_df.iloc[:cut].copy()
        inner_valid = train_df.iloc[cut:].copy()

    return inner_train, inner_valid

def run_fold_lightgbm(sup_df: pd.DataFrame, cutoff_date: pd.Timestamp, horizon_max=12) -> FoldResult:
    cutoff_date = month_start(cutoff_date)

    # Train rows: target_date <= cutoff
    train_df = sup_df[sup_df["target_date"] <= cutoff_date].copy()

    # Inference rows: anchor date == cutoff, horizons 1..H
    infer_df = sup_df[(sup_df["date"] == cutoff_date) & (sup_df["horizon"].astype(int).between(1, horizon_max))].copy()
    if infer_df.empty:
        return FoldResult(cutoff_date=cutoff_date, mape=np.nan, n_predictions=0)

    # Inner validation for early stopping (time-based)
    inner_train, inner_valid = time_inner_split(train_df, date_col="target_date", n_valid_months=3)

    X_tr = inner_train[feature_cols]
    y_tr = inner_train["target"].values

    X_va = inner_valid[feature_cols]
    y_va = inner_valid["target"].values

    # LightGBM categorical features: column indices
    cat_idx = [X_tr.columns.get_loc(c) for c in cat_cols]

    model = train_lightgbm_regressor(X_tr, y_tr, X_va, y_va, categorical_features=cat_idx)

    # Predict
    X_inf = infer_df[feature_cols]
    y_true = infer_df["target"].values
    y_pred = model.predict(X_inf)

    # Post-processing: demand can't be negative
    y_pred = np.clip(y_pred, 0, None)

    mape = safe_mape(y_true, y_pred)
    return FoldResult(cutoff_date=cutoff_date, mape=mape, n_predictions=len(y_pred))

# Run folds
fold_results = [run_fold_lightgbm(sup, c, horizon_max=HORIZON_MAX) for c in cutoffs]
fold_df = pd.DataFrame([r.__dict__ for r in fold_results])

display(fold_df)
if len(fold_df) and fold_df["mape"].notna().any():
    print("Average fold MAPE:", fold_df["mape"].mean())
else:
    print("No valid folds found. Check date range / cutoff logic.")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001050 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3535
[LightGBM] [Info] Number of data points in the train set: 828, number of used features: 74
[LightGBM] [Info] Start training from score 25860.103865
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

,cutoff_date,mape,n_predictions
0,2023-12-01,0.368112,276
1,2024-12-01,0.124917,276


Average fold MAPE: 0.24651451636786914


## 10) Optional: Hyperparameter tuning with Optuna (time-series aware)
This can significantly improve MAPE. It is optional because it can take time.

In [11]:

# -------------------------------
# 10) Optional Optuna tuning
# -------------------------------
DO_TUNE = False  # set True to run

if DO_TUNE:
    try:
        import optuna
    except ImportError:
        !pip -q install optuna
        import optuna

    tune_cutoff = cutoffs[-1] if len(cutoffs) else sup["date"].max()

    def objective(trial):
        params = dict(
            objective="regression",
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            n_estimators=6000,
            num_leaves=trial.suggest_int("num_leaves", 16, 256),
            min_child_samples=trial.suggest_int("min_child_samples", 5, 80),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 0.0, 5.0),
            reg_lambda=trial.suggest_float("reg_lambda", 0.0, 10.0),
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )

        cutoff_date = month_start(tune_cutoff)
        train_df = sup[sup["target_date"] <= cutoff_date].copy()
        infer_df = sup[(sup["date"] == cutoff_date) & (sup["horizon"].astype(int).between(1, HORIZON_MAX))].copy()
        if infer_df.empty:
            return np.inf

        train_df = train_df.sort_values("target_date")
        split_point = train_df["target_date"].quantile(0.90)
        inner_train = train_df[train_df["target_date"] <= split_point]
        inner_valid = train_df[train_df["target_date"] > split_point]

        X_tr = inner_train[feature_cols]
        y_tr = inner_train["target"].values
        X_va = inner_valid[feature_cols]
        y_va = inner_valid["target"].values

        cat_idx = [X_tr.columns.get_loc(c) for c in cat_cols]

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric=lambda y_pred, ds: ("mape", safe_mape(ds.get_label(), y_pred), False),
            categorical_feature=cat_idx,
            callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)]
        )

        y_pred = np.clip(model.predict(infer_df[feature_cols]), 0, None)
        return safe_mape(infer_df["target"].values, y_pred)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=50)

    print("Best MAPE:", study.best_value)
    print("Best params:", study.best_params)


## 11) Train final model at the last available cutoff and forecast next 12 months

In [12]:

# -------------------------------
# 11) Final training and forecasting
# -------------------------------
if len(cutoffs):
    final_cutoff = cutoffs[-1]
else:
    final_cutoff = month_start(df["date"].max() - pd.DateOffset(months=12))

final_cutoff = month_start(final_cutoff)
print("Final cutoff:", final_cutoff.date())

train_df = sup[sup["target_date"] <= final_cutoff].copy()
infer_df = sup[(sup["date"] == final_cutoff) & (sup["horizon"].astype(int).between(1, HORIZON_MAX))].copy()

print("Train rows:", len(train_df), "| Forecast rows:", len(infer_df))
assert len(infer_df) > 0, "No inference rows found for the final cutoff. Check your dates."

# Inner split for early stopping
train_df = train_df.sort_values("target_date")
split_point = train_df["target_date"].quantile(0.90)
inner_train = train_df[train_df["target_date"] <= split_point]
inner_valid = train_df[train_df["target_date"] > split_point]

X_tr = inner_train[feature_cols]
y_tr = inner_train["target"].values
X_va = inner_valid[feature_cols]
y_va = inner_valid["target"].values

cat_idx = [X_tr.columns.get_loc(c) for c in cat_cols]

final_model = train_lightgbm_regressor(X_tr, y_tr, X_va, y_va, categorical_features=cat_idx)

pred = infer_df[["product","aps","series_id","date","horizon","target_date","target"]].copy()
pred["forecast"] = np.clip(final_model.predict(infer_df[feature_cols]), 0, None)

display(pred.head(12))

if pred["target"].notna().any():
    print("Holdout MAPE for final cutoff:", safe_mape(pred["target"].values, pred["forecast"].values))


Final cutoff: 2024-12-01
Train rows: 4830 | Forecast rows: 276
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002293 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6183
[LightGBM] [Info] Number of data points in the train set: 4554, number of used features: 74
[LightGBM] [Info] Start training from score 24588.997365
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

,product,aps,series_id,date,horizon,target_date,target,forecast
23,AH,ACNF,AH|ACNF,2024-12-01,1,2025-01-01,12858.0,12467.683614
58,AH,AH_FIT,AH|AH_FIT,2024-12-01,1,2025-01-01,8070.0,7597.691233
93,AH,AH_R,AH|AH_R,2024-12-01,1,2025-01-01,8051.0,6805.583435
128,AH,ALL,AH|ALL,2024-12-01,1,2025-01-01,61117.0,55226.631915
163,AH,AWUF,AH|AWUF,2024-12-01,1,2025-01-01,8334.0,7236.584189
198,AH,MB,AH|MB,2024-12-01,1,2025-01-01,23804.0,20478.024199
233,CL,ALL,CL|ALL,2024-12-01,1,2025-01-01,24601.0,26661.280334
268,CL,CL,CL|CL,2024-12-01,1,2025-01-01,10452.0,13981.705477
303,CL,CL_FIT,CL|CL_FIT,2024-12-01,1,2025-01-01,4729.0,5208.460668
338,CL,CL_MB,CL|CL_MB,2024-12-01,1,2025-01-01,9420.0,11106.894276


Holdout MAPE for final cutoff: 0.10482592691291653


## 12) Simple ensemble: blend model forecast with a seasonal-naive baseline
Ensembling often stabilizes and can reduce MAPE on strongly seasonal series.

In [13]:

# -------------------------------
# 12) Seasonal-naive ensemble
# -------------------------------
demand_lookup = df[["series_id","date","demand"]].copy()

pred_ens = pred.merge(
    demand_lookup.rename(columns={"date":"target_date", "demand":"seasonal_naive"}),
    on=["series_id","target_date"],
    how="left",
)

w_model = 0.8
w_naive = 0.2

pred_ens["forecast_ensemble"] = (
    w_model * pred_ens["forecast"] + w_naive * pred_ens["seasonal_naive"].fillna(pred_ens["forecast"])
)
pred_ens["forecast_ensemble"] = np.clip(pred_ens["forecast_ensemble"], 0, None)

display(pred_ens.head(12))

if pred_ens["target"].notna().any():
    print("MAPE (model):", safe_mape(pred_ens["target"].values, pred_ens["forecast"].values))
    print("MAPE (ensemble):", safe_mape(pred_ens["target"].values, pred_ens["forecast_ensemble"].values))


,product,aps,series_id,date,horizon,target_date,target,forecast,seasonal_naive,forecast_ensemble
0,AH,ACNF,AH|ACNF,2024-12-01,1,2025-01-01,12858.0,12467.683614,12858,12545.746891
1,AH,AH_FIT,AH|AH_FIT,2024-12-01,1,2025-01-01,8070.0,7597.691233,8070,7692.152987
2,AH,AH_R,AH|AH_R,2024-12-01,1,2025-01-01,8051.0,6805.583435,8051,7054.666748
3,AH,ALL,AH|ALL,2024-12-01,1,2025-01-01,61117.0,55226.631915,61117,56404.705532
4,AH,AWUF,AH|AWUF,2024-12-01,1,2025-01-01,8334.0,7236.584189,8334,7456.067351
5,AH,MB,AH|MB,2024-12-01,1,2025-01-01,23804.0,20478.024199,23804,21143.219360
6,CL,ALL,CL|ALL,2024-12-01,1,2025-01-01,24601.0,26661.280334,24601,26249.224267
7,CL,CL,CL|CL,2024-12-01,1,2025-01-01,10452.0,13981.705477,10452,13275.764382
8,CL,CL_FIT,CL|CL_FIT,2024-12-01,1,2025-01-01,4729.0,5208.460668,4729,5112.568535
9,CL,CL_MB,CL|CL_MB,2024-12-01,1,2025-01-01,9420.0,11106.894276,9420,10769.515421


MAPE (model): 0.10482592691291653
MAPE (ensemble): 0.08386074153033322


## 13) Optional: Prediction intervals (Quantile LightGBM)
If you want uncertainty, train quantile models for lower/upper bounds.

In [14]:

# -------------------------------
# 13) Quantile regression (optional)
# -------------------------------
DO_QUANTILES = False  # set True if you want intervals

if DO_QUANTILES:
    def train_lgb_quantile(alpha):
        params = dict(
            objective="quantile",
            alpha=alpha,
            learning_rate=0.03,
            n_estimators=5000,
            num_leaves=63,
            min_child_samples=20,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            categorical_feature=cat_idx,
            callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)]
        )
        return model

    q_lo = train_lgb_quantile(0.10)
    q_hi = train_lgb_quantile(0.90)

    pred_int = pred_ens.copy()
    pred_int["p10"] = np.clip(q_lo.predict(infer_df[feature_cols]), 0, None)
    pred_int["p90"] = np.clip(q_hi.predict(infer_df[feature_cols]), 0, None)

    display(pred_int[["series_id","target_date","forecast_ensemble","p10","p90"]].head(12))


## 14) Optional: Explainability with SHAP
This helps you justify *why* the forecast moved (industry-grade requirement).

In [15]:

# -------------------------------
# 14) SHAP explainability (optional)
# -------------------------------
DO_SHAP = False  # set True to compute SHAP

if DO_SHAP:
    try:
        import shap
    except ImportError:
        !pip -q install shap
        import shap

    sample = inner_train.sample(min(3000, len(inner_train)), random_state=RANDOM_SEED)
    X_sample = sample[feature_cols]

    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_sample)

    shap.summary_plot(shap_values, X_sample, max_display=25)


## 15) Export forecasts
Create both a long-format and a wide-format (Jan..Dec) file for easy submission.

In [16]:

# -------------------------------
# 15) Export
# -------------------------------
out_long = pred_ens[["product","aps","series_id","date","horizon","target_date","forecast","forecast_ensemble"]].copy()
out_long = out_long.sort_values(["product","aps","target_date"]).reset_index(drop=True)

display(out_long.head(12))

out_long["year"] = out_long["target_date"].dt.year
out_long["month"] = out_long["target_date"].dt.strftime("%b")
out_long["month"] = pd.Categorical(out_long["month"], categories=MONTHS, ordered=True)

wide = (
    out_long.pivot_table(
        index=["product","aps","year"],
        columns="month",
        values="forecast_ensemble",
        aggfunc="first",
    )
    .reset_index()
)

for m in MONTHS:
    if m not in wide.columns:
        wide[m] = np.nan
wide = wide[["product","aps","year"] + MONTHS]

display(wide.head())

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

long_path = OUTPUT_DIR / "forecast_long.csv"
wide_path = OUTPUT_DIR / "forecast_wide.csv"

out_long.to_csv(long_path, index=False)
wide.to_csv(wide_path, index=False)

print("Saved:")
print(" -", long_path.resolve())
print(" -", wide_path.resolve())


,product,aps,series_id,date,horizon,target_date,forecast,forecast_ensemble
0,AH,ACNF,AH|ACNF,2024-12-01,1,2025-01-01,12467.683614,12545.746891
1,AH,ACNF,AH|ACNF,2024-12-01,2,2025-02-01,13455.875745,12923.300596
2,AH,ACNF,AH|ACNF,2024-12-01,3,2025-03-01,14974.415713,14590.332570
3,AH,ACNF,AH|ACNF,2024-12-01,4,2025-04-01,16346.590765,16307.072612
4,AH,ACNF,AH|ACNF,2024-12-01,5,2025-05-01,17108.394598,17289.715679
5,AH,ACNF,AH|ACNF,2024-12-01,6,2025-06-01,17359.318626,17468.454901
6,AH,ACNF,AH|ACNF,2024-12-01,7,2025-07-01,16901.001596,17484.201277
7,AH,ACNF,AH|ACNF,2024-12-01,8,2025-08-01,16069.412549,16370.530039
8,AH,ACNF,AH|ACNF,2024-12-01,9,2025-09-01,14196.574035,14915.259228
9,AH,ACNF,AH|ACNF,2024-12-01,10,2025-10-01,13280.670051,13816.936041


month,product,aps,year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,AH,ACNF,2025,12545.746891,12923.300596,14590.332570,16307.072612,17289.715679,17468.454901,17484.201277,16370.530039,14915.259228,13816.936041,12917.080233,12817.486060
1,AH,AH_FIT,2025,7692.152987,8705.514086,8517.094102,10247.502058,11210.230443,11757.784600,12324.038081,11708.808577,11063.936382,9709.157383,8782.484936,8467.683247
2,AH,AH_R,2025,7054.666748,7595.173512,8585.816229,9769.968836,10486.143195,10816.098382,11045.073568,10234.109342,9467.556499,8441.899452,7763.635945,7713.234879
3,AH,ALL,2025,56404.705532,59634.097109,64782.567259,73821.286090,83185.162924,85251.396310,89305.501902,84235.921737,77256.339805,69255.509419,61011.728472,56979.572166
4,AH,AWUF,2025,7456.067351,7757.620341,8701.471900,9882.129668,10403.109126,10652.235896,10954.544808,10134.692499,9184.518186,8592.582640,7877.753934,7898.827761


Saved:
 - D:\Projects\@Scale AI Hackathon\Codes\outputs\forecast_long.csv
 - D:\Projects\@Scale AI Hackathon\Codes\outputs\forecast_wide.csv


In [ ]:
import json
from pathlib import Path
import joblib

OUTPUT_DIR = Path("./models")
OUTPUT_DIR.mkdir(exist_ok=True)

# (A) Save as a joblib "bundle" (recommended: includes metadata)
bundle = {
    "model": final_model,          # LightGBM sklearn wrapper
    "feature_cols": feature_cols,
    "cat_cols": cat_cols,
    "months": MONTHS,
    "month_map": MONTH_MAP,
    "horizon_max": HORIZON_MAX,
    "random_seed": RANDOM_SEED,
    # ensemble weights (if you use the blend cell)
    "ensemble": {
        "use": True,
        "w_model": 0.8,
        "w_naive": 0.2
    }
}

bundle_path = OUTPUT_DIR / "demand_forecasting_model_bundle.joblib"
joblib.dump(bundle, bundle_path)

print("Saved bundle to:", bundle_path.resolve())

# (B) Also save native LightGBM booster for maximum portability
booster_path = OUTPUT_DIR / "lgbm_booster.txt"
final_model.booster_.save_model(str(booster_path))
print("Saved booster to:", booster_path.resolve())


Saved bundle to: D:\Projects\@Scale AI Hackathon\Codes\models\p1_lgbm_model_bundle.joblib
Saved booster to: D:\Projects\@Scale AI Hackathon\Codes\models\lgbm_booster.txt


In [18]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
MONTH_MAP = {m: i+1 for i, m in enumerate(MONTHS)}

def month_start(dt):
    return pd.to_datetime(dt).to_period("M").to_timestamp()

def melt_monthly_data(df_wide: pd.DataFrame) -> pd.DataFrame:
    id_cols = [c for c in df_wide.columns if c not in MONTHS]
    df_long = df_wide.melt(
        id_vars=id_cols,
        value_vars=MONTHS,
        var_name="month",
        value_name="demand",
    )
    df_long["month_num"] = df_long["month"].map(MONTH_MAP).astype(int)
    df_long["date"] = pd.to_datetime(df_long[["year"]].assign(month=df_long["month_num"], day=1))
    df_long = df_long.sort_values(["product", "aps", "date"]).reset_index(drop=True)
    return df_long

def build_base_panel(data_dir: Path):
    demand_wide = pd.read_csv(data_dir / "demand_history.csv")
    demand = melt_monthly_data(demand_wide)

    df = demand.copy()
    df["series_id"] = df["product"].astype(str) + "|" + df["aps"].astype(str)
    df["date"] = df["date"].dt.to_period("M").dt.to_timestamp()
    df["year"] = df["date"].dt.year
    df["month_num"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["month_sin"] = np.sin(2 * np.pi * df["month_num"] / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * df["month_num"] / 12.0)

    # external signals (optional)
    ext_path = data_dir / "external_signals.csv"
    if ext_path.exists():
        external = pd.read_csv(ext_path)

        if "date" in external.columns:
            external["date"] = pd.to_datetime(external["date"]).dt.to_period("M").dt.to_timestamp()
        elif {"year", "month"}.issubset(external.columns):
            external["date"] = pd.to_datetime(external[["year"]].assign(month=external["month"], day=1))
        elif {"year", "month_num"}.issubset(external.columns):
            external["date"] = pd.to_datetime(external[["year"]].assign(month=external["month_num"], day=1))
        else:
            raise ValueError("external_signals.csv must contain either 'date' or ('year','month') columns.")

        external = external.sort_values("date").reset_index(drop=True)
        ignore_cols = {"date", "year", "month", "month_num"}
        exog_cols = [c for c in external.columns if c not in ignore_cols and pd.api.types.is_numeric_dtype(external[c])]

        EXOG_LAGS = [0, 1, 3, 6, 12]
        for col in exog_cols:
            for lag in EXOG_LAGS:
                external[f"{col}_lag{lag}"] = external[col].shift(lag)

        keep_cols = ["date"] + [c for c in external.columns if c.endswith(tuple([f"_lag{l}" for l in EXOG_LAGS]))]
        external_feat = external[keep_cols].copy()

        df = df.merge(external_feat, on="date", how="left")

    return df

def add_lag_roll_features(df: pd.DataFrame):
    df = df.sort_values(["series_id", "date"]).reset_index(drop=True)
    g = df.groupby("series_id", sort=False)

    # demand lags
    for lag in range(0, 13):
        df[f"demand_lag{lag}"] = g["demand"].shift(lag)

    # rolling stats
    for w in [3, 6, 12]:
        df[f"demand_roll_mean_{w}"] = g["demand"].rolling(window=w, min_periods=max(2, w//2)).mean().reset_index(level=0, drop=True)
        df[f"demand_roll_std_{w}"]  = g["demand"].rolling(window=w, min_periods=max(2, w//2)).std().reset_index(level=0, drop=True)

    df["demand_diff_1"] = df["demand"] - df["demand_lag1"]
    df["demand_diff_12"] = df["demand"] - df["demand_lag12"]
    return df

def make_direct_multi_horizon(base_df: pd.DataFrame, horizon_max=12):
    out = []
    gb = base_df.groupby("series_id", sort=False)
    for h in range(1, horizon_max+1):
        tmp = base_df.copy()
        tmp["horizon"] = h
        tmp["target_date"] = tmp["date"] + pd.DateOffset(months=h)
        out.append(tmp)
    sup = pd.concat(out, axis=0, ignore_index=True)
    return sup

def forecast_12_months(data_dir: str, model_bundle_path: str, cutoff_date: str):
    data_dir = Path(data_dir)
    cutoff = month_start(cutoff_date)

    bundle = joblib.load(model_bundle_path)
    model = bundle["model"]
    feature_cols = bundle["feature_cols"]
    cat_cols = bundle["cat_cols"]
    w_model = bundle.get("ensemble", {}).get("w_model", 1.0)
    w_naive = bundle.get("ensemble", {}).get("w_naive", 0.0)

    df = build_base_panel(data_dir)
    df = add_lag_roll_features(df)

    # Build supervised anchors at cutoff
    sup = make_direct_multi_horizon(df, horizon_max=bundle.get("horizon_max", 12))

    # Keep only the anchor month = cutoff
    infer_df = sup[sup["date"] == cutoff].copy()
    infer_df = infer_df[infer_df["horizon"].between(1, bundle.get("horizon_max", 12))].copy()

    # Convert categoricals same as training expectation
    for c in ["product","aps","series_id","horizon","month_num","quarter"]:
        if c in infer_df.columns:
            infer_df[c] = infer_df[c].astype("category")

    # Ensure all feature columns exist (if some exog columns are missing)
    for c in feature_cols:
        if c not in infer_df.columns:
            infer_df[c] = np.nan

    X_inf = infer_df[feature_cols]
    pred_model = np.clip(model.predict(X_inf), 0, None)

    # Seasonal naive baseline computed ONLY from history at cutoff:
    # seasonal_naive(h) = demand at (cutoff - (12-h) months) == demand_lag(12-h)
    lag_idx = 12 - infer_df["horizon"].astype(int)
    seasonal = np.array([infer_df.iloc[i][f"demand_lag{lag_idx.iloc[i]}"] for i in range(len(infer_df))], dtype=float)

    # Blend (optional)
    pred_ens = w_model * pred_model + w_naive * np.nan_to_num(seasonal, nan=pred_model)

    out = infer_df[["product","aps","series_id","date","horizon","target_date"]].copy()
    out["forecast_model"] = pred_model
    out["forecast_ensemble"] = pred_ens
    out = out.sort_values(["series_id","target_date"]).reset_index(drop=True)
    return out

# Example usage:
# out = forecast_12_months(
#     data_dir="../data",
#     model_bundle_path="./outputs/lgbm_model_bundle.joblib",
#     cutoff_date="2024-12-01"
# )
# display(out.head(12))
